In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree
import random
from torch.utils.data import DataLoader
from tqdm import tqdm

# --- 1. Granular Ball & SegGBC Classes ---
class GrandularBall3D:
    def __init__(self, points):
        self.points = np.array(points, dtype=np.float64)
        self.dim = self.points.shape[1]
        self.center = np.mean(self.points, axis=0)
        self.radius_max = 0.0
        self.radius_avg = 0.0
        self.radius_sta = 0.0
        self.mu = None
        self.nu = None
        self.pi = None
        self.scm = 1.0

    def compute_intuitionistic_fuzzy_parameters(self, alpha=0.7):
        n = len(self.points)
        if n == 0:
            return 
        diffs = self.points - self.center
        dist_sq = np.sum(diffs ** 2, axis=1)
        r_max_temp = np.max(dist_sq) if n > 1 else 1e-5
        sigma_m = max(alpha * np.sqrt(r_max_temp + 1e-8), 1e-5)
        sigma_n = max((1.0 - alpha) * np.sqrt(r_max_temp + 1e-8), 1e-5)

        self.mu = np.exp(-dist_sq / (sigma_m ** 2))
        self.nu = 1.0 - np.exp(-dist_sq / (sigma_n ** 2))
        self.pi = np.clip(1.0 - self.mu - self.nu, 0.0, 1.0)
        
        theta_pi = self.pi[:, np.newaxis]
        self.center = np.mean(self.points + theta_pi, axis=0)

        diffs = self.points - self.center
        dist_pi = np.sum(diffs ** 2, axis=1) + self.pi

        self.radius_max = np.max(dist_pi) if n > 1 else 1e-5
        self.radius_avg = np.mean(dist_pi) if n > 1 else 1e-5
        self.radius_sta = self.radius_avg * 0.5


# --- 2. Integrated Pipeline Runner ---
def run_unet_seg_gbc_with_auv(
    unet_model: nn.Module, 
    image_tensor: torch.Tensor, 
    seg_gbc_instance, 
    device: torch.device,
    show_progress: bool = False,
):
    """
    1. Passes image through UNet encoder-decoder to get high-res spatial feature map.
    2. Runs SegGBC to partition feature space into granular balls.
    3. Computes SVD-based AUV entropy per granular ball.
    """
    unet_model.eval()
    if image_tensor.ndim == 3:
        image_tensor = image_tensor.unsqueeze(0)
    image_tensor = image_tensor.to(device)

    with torch.no_grad():
        # Assume UNet returns (segmentation_logits, upsampled_spatial_features [B, C, H, W])
        logits, spatial_features = unet_model(image_tensor)

    # Convert features to numpy for SegGBC: shape [H, W, C]
    feat_np = spatial_features.squeeze(0).permute(1, 2, 0).cpu().numpy()
    H, W, C = feat_np.shape

    # Run SegGBC clustering on the high-res latent feature map
    spatial_weight = 0.1
    yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
    
    # Normalize features
    feat_scales = np.max(np.abs(feat_np), axis=(0, 1), keepdims=True)
    feat_scales[feat_scales == 0] = 1.0
    norm_feat = feat_np / feat_scales

    features_for_gbc = np.column_stack([
        norm_feat.reshape(-1, C),
        xx.reshape(-1).astype(np.float64) / W * spatial_weight,
        yy.reshape(-1).astype(np.float64) / H * spatial_weight,
    ])

    # Initialize and fit SegGBC granular balls
    initial_ball = GrandularBall3D(features_for_gbc)
    initial_ball.compute_intuitionistic_fuzzy_parameters(alpha=0.7)
    seg_gbc_instance.fit(
        initial_ball.points.reshape(H, W, -1),
        show_progress=show_progress,
    )
    
    # --- Step 3: Compute AUV per Granular Ball via SVD Spectral Entropy ---
    ball_auv_scores = []
    for ball in seg_gbc_instance.balls:
        if len(ball.points) < 2:
            ball_auv_scores.append(1.0)
            continue
            
        # Extract feature sub-matrix for points inside this granular ball (excluding spatial coords)
        ball_feats = torch.tensor(ball.points[:, :C], dtype=torch.float32)
        ball_feats = ball_feats - ball_feats.mean(dim=0, keepdim=True)
        
        # SVD singular values
        try:
            singular_values = torch.linalg.svdvals(ball_feats)
            energy = singular_values.square()
            energy_sum = energy.sum()
            rank = energy.numel()
            
            if rank <= 1 or energy_sum <= 1e-12:
                auv_score = 0.0
            else:
                probabilities = (energy / energy_sum).clamp_min(1e-12)
                entropy = -(probabilities * probabilities.log()).sum()
                auv_score = (entropy / np.log(rank)).item()
        except Exception:
            auv_score = 0.5
            
        ball_auv_scores.append(auv_score)

    return logits, seg_gbc_instance.balls, ball_auv_scores

In [2]:
class SegGBC:
    def __init__(self, beta=0.8, epsilon=0.75, alpha=0.7):
        self.beta = beta
        self.epsilon = epsilon
        self.alpha = alpha
        self.balls = []

    def fit(self, image, spatial_weight=0.1, show_progress=True):
        if image.ndim != 3:
            raise ValueError(f"Expected image shape (H, W, C); got {getattr(image, 'shape', None)}")
        if image.shape[-1] < 1:
            raise ValueError(f"Expected at least one feature channel; got {image.shape}")
        if spatial_weight <=0:
            raise ValueError("spatial_weight must be greater than 0")

        H,W,C = image.shape

        if show_progress:
            print("constructing 2d feature space...")
        # Scale each modality without assuming the image values are 0–255.
        image = image.astype(np.float64)
        scales = np.max(np.abs(image), axis=(0, 1), keepdims=True)
        scales[scales == 0] = 1.0
        normalized_image = image / scales

        yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")

        features = np.column_stack([
            normalized_image.reshape(-1, C),
            xx.reshape(-1).astype(np.float64) / W * spatial_weight,
            yy.reshape(-1).astype(np.float64) / H * spatial_weight,
        ])

        intital_ball = GrandularBall3D(features)
        intital_ball.compute_intuitionistic_fuzzy_parameters(self.alpha)
        self.balls = [intital_ball]

        
        iteration = 0
        max_iter = 30

        with tqdm(total=max_iter, desc="Splitting Progress", unit='iter', disable=not show_progress) as pbar:
            while iteration < max_iter:
                iteration +=1
                new_balls = []
                split_occurred = False

                for ball in self.balls:
                    ball.compute_intuitionistic_fuzzy_parameters(self.alpha)
                    ball.radius_sta = ball.radius_avg * 0.5

                    d = ball.dim
                    r_max_vol = max(ball.radius_max ** d, 1e-8)
                    r_avg_vol = max(ball.radius_avg ** d, 1e-8)
                    r_sta_vol = max(ball.radius_sta ** d, 1e-8)

                    rho_max = len(ball.points) / r_max_vol

                    diffs = ball.points - ball.center
                    dist_pi = np.sum(diffs ** 2, axis=1) + ball.pi

                    rho_avg = np.sum(dist_pi <= ball.radius_avg) / r_avg_vol
                    rho_sta = np.sum(dist_pi <= ball.radius_sta) / r_sta_vol

                    num = min(rho_avg,rho_max) * min(rho_sta, rho_max)
                    den = max(rho_avg, rho_max) * max(rho_sta, rho_max) + 1e-8

                    ball.scm = num / den

                    if ball.scm < self.beta and len(ball.points) > 20:
                        split_occurred = True
                        mid = len(ball.points) // 2
                        new_balls.extend([
                            GrandularBall3D(ball.points[:mid]),
                            GrandularBall3D(ball.points[mid:])
                        ])
                    else:
                        new_balls.append(ball)

                self.balls = new_balls

                pbar.set_postfix(balls_count=len(self.balls))
                pbar.update(1)
                if not split_occurred:
                    break
        
        self._merge_clusters(show_progress=show_progress)

        
        seg_mask = self._generate_2D_mask(H, W, spatial_weight)
        return seg_mask

    def _merge_clusters(self, show_progress=True):
        if len(self.balls) <=1:
            return

        

        centers = np.array([b.center for b in self.balls])
        tree = cKDTree(centers)

        avg_radius = np.mean([b.radius_max for b in self.balls])
        neighbour_radius = max(avg_radius * 3.0, 1.0)

        pairs = tree.query_pairs(r=neighbour_radius)

        merged_indices = set()

        with tqdm(pairs, desc="Merging Progress", unit="pairs", disable=not show_progress) as pbar:
            for i,j in pbar:
                if i in merged_indices or j in merged_indices:
                    continue

                b1, b2 = self.balls[i], self.balls[j]

                dist_centers = np.linalg.norm(b1.center - b2.center)
                sim = 1.0/(1.0 + dist_centers)

                if sim > self.epsilon:
                    merged_indices.add(j)
                    b1.points = np.vstack([b1.points, b2.points])
                    b1.compute_intuitionistic_fuzzy_parameters(self.alpha)
                    pbar.set_postfix(merged=len(merged_indices))

        initial_count = len(self.balls)
        self.balls = [b for idx,b in enumerate(self.balls) if idx not in merged_indices]
        

    def _generate_2D_mask(self, H, W, spatial_weight):
        mask = np.zeros((H, W), dtype=np.int32)

        for cluster_id, ball in enumerate(self.balls):
            for pt in ball.points:
                x = int(round(pt[4] * W / spatial_weight))
                y = int(round(pt[5] * H / spatial_weight))

                if 0 <= y < H and 0 <= x < W:
                    mask[y, x] = cluster_id

        return mask
    




In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class ResNetSegmentationWrapper(nn.Module):
    def __init__(self, num_classes=4, in_channels=4):
        super().__init__()
        # Load an original ResNet (e.g., ResNet50)
        resnet = models.resnet50(weights=None)
        
        # Modify the first conv layer to accept 4-channel MRI input instead of 3-channel RGB
        self.in_conv = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.in_conv.weight.data[:, :3, :, :] = resnet.conv1.weight.data  # Copy RGB weights
        self.in_conv.weight.data[:, 3:4, :, :] = resnet.conv1.weight.data[:, :1, :, :]  # Duplicate for 4th channel
        
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        
        # ResNet encoder layers
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4  # Output: [Batch, 2048, H/32, W/32]
        
        # Simple Segmentation Header / Upsampling path to get back to [Batch, num_classes, H, W]
        self.seg_head = nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, num_classes, kernel_size=1)
        )

    def forward(self, x):
        input_size = x.shape[-2:]  # e.g., (240, 240)
        
        # Encoder path
        x = self.in_conv(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        spatial_latent = self.layer4(x)  # Shape: [Batch, 2048, 8, 8]
        
        # Decode / Upsample back to original image size
        logits = self.seg_head(spatial_latent)
        logits = F.interpolate(logits, size=input_size, mode='bilinear', align_corners=False)
        
        # Returns both the pixel logits (for CrossEntropyLoss) and latent features (for SegGBC / AUV)
        return logits, spatial_latent

In [4]:
import h5py
import numpy as np
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader

class BraTSSliceDataset(Dataset):
    def __init__(self, files: list[Path]) -> None:
        self.files = list(files)

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, str]:
        path = self.files[index]

        with h5py.File(path, "r") as h5_file:
            image = np.asarray(h5_file["image"][...], dtype=np.float32)
            mask = np.asarray(h5_file["mask"][...], dtype=np.float32)

        # 1. Image normalization
        scale = np.max(np.abs(image), axis=(0, 1), keepdims=True)
        scale[scale < 1e-6] = 1.0
        image = np.clip(image / scale, -1.0, 1.0)
        image_tensor = torch.from_numpy(image).permute(2, 0, 1)

        # 2. Fix Mask Shape for CrossEntropyLoss
        # If mask is stored as [H, W, C] (multi-channel binary or one-hot):
        if mask.ndim == 3:
            # Convert multi-channel binary mask to a single class index map [H, W]
            # Example: argmax across channels gives class indices 0, 1, 2...
            mask = np.argmax(mask, axis=-1).astype(np.int64)
        else:
            mask = mask.astype(np.int64)

        # Resulting mask tensor shape: [H, W] with integer class labels
        mask_tensor = torch.from_numpy(mask)

        return image_tensor, mask_tensor, str(path)

In [5]:
device = torch.device("cuda")
data_dir = Path("data/3/BraTS2020_training_data/content/data")

h5_files = list(data_dir.glob("*.h5"))

# Optional: If your filenames contain patient IDs (e.g., "BraTS20_Training_001_slice12.h5"),
# group them by patient first to ensure no patient overlap between sets.

# 2. Shuffle and Split (e.g., 70% Train, 15% Validation, 15% Test)
random.seed(42)  # For reproducibility
random.shuffle(h5_files)

total_files = len(h5_files)
train_split = int(0.70 * total_files)
val_split = int(0.15 * total_files)

train_files = h5_files[:train_split]
val_files = h5_files[train_split:train_split + val_split]
test_files = h5_files[train_split + val_split:]

print(f"Total files: {total_files}")
print(f"Train set: {len(train_files)} files")
print(f"Val set:   {len(val_files)} files")
print(f"Test set:  {len(test_files)} files")

# 3. Create separate Dataset and DataLoader instances for each split
train_dataset = BraTSSliceDataset(files=train_files)
val_dataset = BraTSSliceDataset(files=val_files)
test_dataset = BraTSSliceDataset(files=test_files)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

Total files: 57195
Train set: 40036 files
Val set:   8579 files
Test set:  8580 files


In [6]:
num_classes = 4  # Background, NCR/NET, Edema, Enhancing Tumor
model = ResNetSegmentationWrapper(num_classes=num_classes, in_channels=4).to(device)

# 5. Train the segmentation model
num_epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)
latest_checkpoint = checkpoint_dir / "resnet_segmentation_latest.pth"
final_model_path = Path("brats_resnet_final.pth")
start_epoch = 1

if latest_checkpoint.exists():
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    print(f"Found checkpoint; resuming from epoch {start_epoch}.")
else:
    print(f"No checkpoint found at {latest_checkpoint}; starting from scratch.")

epoch_progress = tqdm(range(start_epoch, num_epochs + 1), desc="Training", unit="epoch")
for epoch in epoch_progress:
    model.train()
    running_loss = 0.0

    for images, masks, _ in train_loader:
        images = images.to(device)
        masks = masks.to(device=device, dtype=torch.long)

        optimizer.zero_grad()
        logits, _ = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        epoch_progress.set_postfix(train_loss=f"{running_loss / len(train_loader):.4f}")

    epoch_loss = running_loss / len(train_loader)
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": epoch_loss,
    }
    temp_checkpoint = latest_checkpoint.with_suffix(".tmp")
    torch.save(checkpoint, temp_checkpoint)
    temp_checkpoint.replace(latest_checkpoint)
    epoch_progress.write(f"Saved checkpoint: {latest_checkpoint}")

torch.save(model.state_dict(), final_model_path)
print(f"Saved final model weights to {final_model_path}")

# 5. Initialize SegGBC Instance
#   # Ensure SegGBC class is imported
seg_gbc = SegGBC(beta=0.8, epsilon=0.75, alpha=0.7)

# 6. Run the Pipeline over Training Files
print("\n--- Starting U-Net + SegGBC + AUV Pipeline Execution ---")

results_summary = {}

total_batches = len(train_loader)
for batch_idx, (images, masks, paths) in enumerate(
    tqdm(train_loader, total=total_batches, desc="Pipeline", unit="batch")
):
    file_path = paths[0]
    file_name = Path(file_path).name

    single_image = images[0]  # Shape: [4, 240, 240]

    try:
        # Execute your integrated pipeline function
        logits, balls, ball_auv_scores = run_unet_seg_gbc_with_auv(
            unet_model=model,
            image_tensor=single_image,
            seg_gbc_instance=seg_gbc,
            device=device,
            show_progress=False,
        )

        mean_auv = float(np.mean(ball_auv_scores)) if ball_auv_scores else 0.0

        results_summary[file_name] = {
            "num_balls": len(balls),
            "mean_auv": mean_auv,
            "logits_shape": logits.shape
        }
    except Exception as e:
        tqdm.write(f"Error processing {file_name}: {e}")


print(f"\nPipeline finished: {len(results_summary)}/{total_batches} batches completed.")

No checkpoint found at checkpoints/resnet_segmentation_latest.pth; starting from scratch.


Training:  10%|█         | 1/10 [09:20<1:24:04, 560.53s/epoch, train_loss=0.0240]

Saved checkpoint: checkpoints/resnet_segmentation_latest.pth


Training:  20%|██        | 2/10 [15:55<1:01:44, 463.06s/epoch, train_loss=0.0142]

Saved checkpoint: checkpoints/resnet_segmentation_latest.pth


Training:  30%|███       | 3/10 [20:30<43:59, 377.12s/epoch, train_loss=0.0128]  

... (13 outputs left out)

Training: 100%|██████████| 10/10 [48:52<00:00, 293.30s/epoch, train_loss=0.0109]


Saved checkpoint: checkpoints/resnet_segmentation_latest.pth
Saved final model weights to brats_resnet_final.pth

--- Starting U-Net + SegGBC + AUV Pipeline Execution ---


Pipeline:   0%|          | 0/10009 [00:00<?, ?batch/s]/tmp/ipykernel_2791917/3839851239.py:53: RuntimeWarning: overflow encountered in scalar power
  r_max_vol = max(ball.radius_max ** d, 1e-8)
/tmp/ipykernel_2791917/3839851239.py:54: RuntimeWarning: overflow encountered in scalar power
  r_avg_vol = max(ball.radius_avg ** d, 1e-8)
/tmp/ipykernel_2791917/3839851239.py:55: RuntimeWarning: overflow encountered in scalar power
  r_sta_vol = max(ball.radius_sta ** d, 1e-8)
Pipeline: 100%|██████████| 10009/10009 [04:53<00:00, 34.13batch/s]


Pipeline finished: 10009/10009 batches completed.


In [9]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from pathlib import Path

def test_model_pipeline(
    model: torch.nn.Module,
    test_files: list[Path],
    device: torch.device,
    num_classes: int = 4,  # Background, NCR/NET, ED, ET
    num_visualization_samples: int = 5,
):
    """
    Evaluates the model on a dedicated test dataset, computing per-class 
    and mean Dice/IoU scores along with qualitative predictions.
    """
    model.eval()
    
    # 1. Initialize test loader (shuffle=False for deterministic evaluation order)
    test_dataset = BraTSSliceDataset(files=test_files)
    test_loader = DataLoader(
        test_dataset, 
        batch_size=1, 
        shuffle=False, 
        num_workers=2
    )

    # Accumulators for global confusion metrics per class
    total_intersection = np.zeros(num_classes, dtype=np.float64)
    total_union = np.zeros(num_classes, dtype=np.float64)
    total_target_area = np.zeros(num_classes, dtype=np.float64)
    total_pred_area = np.zeros(num_classes, dtype=np.float64)

    print(f"Running inference and evaluation on {len(test_files)} test files...")

    results_log = []
    qualitative_samples = []

    with torch.inference_mode():
        for batch_idx, (images, masks, paths) in enumerate(test_loader):
            images = images.to(device)
            masks = masks.to(device).long()  # Shape: [1, H, W]
            file_name = Path(paths[0]).name

            # This model returns (segmentation_logits, spatial_features).
            outputs = model(images)
            if isinstance(outputs, dict):
                logits = outputs["out"]
            elif isinstance(outputs, (tuple, list)):
                logits = outputs[0]
            else:
                logits = outputs
            if not isinstance(logits, torch.Tensor) or logits.ndim != 4:
                raise ValueError("Expected model logits with shape [batch, classes, height, width].")
            if logits.shape[1] != num_classes:
                raise ValueError(f"Model returned {logits.shape[1]} classes; expected {num_classes}.")

            # Argmax to get predicted class mask: [1, H, W] -> [H, W]
            preds = torch.argmax(logits, dim=1)

            preds_np = preds.cpu().numpy()[0]
            masks_np = masks.cpu().numpy()[0]

            # Compute slice-level metrics
            slice_dice = {}
            for c in range(num_classes):
                pred_c = (preds_np == c)
                mask_c = (masks_np == c)

                intersection = np.logical_and(pred_c, mask_c).sum()
                union = np.logical_or(pred_c, mask_c).sum()
                
                target_sum = mask_c.sum()
                pred_sum = pred_c.sum()

                total_intersection[c] += intersection
                total_union[c] += union
                total_target_area[c] += target_sum
                total_pred_area[c] += pred_sum

                # Dice for this class on this specific slice
                denom = pred_sum + target_sum
                slice_dice[c] = (2.0 * intersection / denom) if denom > 0 else 1.0

            results_log.append({
                "file": file_name,
                "slice_dice": slice_dice
            })
            if len(qualitative_samples) < num_visualization_samples:
                qualitative_samples.append({
                    "file": file_name,
                    "pred_mask": preds_np,
                    "true_mask": masks_np
                })

    # --- Compute Global Test Metrics ---
    class_dice = np.zeros(num_classes)
    class_iou = np.zeros(num_classes)

    print("\n" + "="*40)
    print("FINAL TEST SET EVALUATION RESULTS")
    print("="*40)
    
    class_names = ["Background (0)", "NCR/NET (1)", "Edema (2)", "Enhancing Tumor (3)"]
    
    for c in range(num_classes):
        denom_dice = total_pred_area[c] + total_target_area[c]
        class_dice[c] = (2.0 * total_intersection[c] / denom_dice) if denom_dice > 0 else 1.0
        class_iou[c] = (total_intersection[c] / total_union[c]) if total_union[c] > 0 else 1.0
        
        print(f"{class_names[c]:<25} | Dice: {class_dice[c]:.4f} | IoU: {class_iou[c]:.4f}")

    # Mean tumor score (excluding background class index 0)
    mean_dice = np.mean(class_dice[1:])
    mean_iou = np.mean(class_iou[1:])

    print("-" * 40)
    print(f"{'Mean Tumor Score':<25} | Dice: {mean_dice:.4f} | IoU: {mean_iou:.4f}")
    print("="*40)

    return {
        "class_dice": class_dice,
        "class_iou": class_iou,
        "mean_dice": mean_dice,
        "mean_iou": mean_iou,
        "detailed_results": results_log,
        "qualitative_samples": qualitative_samples
    }


final_model_path = Path("brats_resnet_final.pth")
if final_model_path.exists():
    model.load_state_dict(torch.load(final_model_path, map_location=device))
    print(f"Loaded trained weights from {final_model_path}")
else:
    print(f"{final_model_path} not found; testing the model currently in memory.")

test_results = test_model_pipeline(
    model=model,
    test_files=test_files,
    device=device,
    num_classes=4
)


Loaded trained weights from brats_resnet_final.pth
Running inference and evaluation on 8580 test files...

FINAL TEST SET EVALUATION RESULTS
Background (0)            | Dice: 0.9977 | IoU: 0.9954
NCR/NET (1)               | Dice: 0.6937 | IoU: 0.5311
Edema (2)                 | Dice: 0.6019 | IoU: 0.4305
Enhancing Tumor (3)       | Dice: 1.0000 | IoU: 1.0000
----------------------------------------
Mean Tumor Score          | Dice: 0.7652 | IoU: 0.6539
